
# NeoLM — Inference

Load the released [NeoLM-Small-Base](https://huggingface.co/MaagDeveloper/NeoLM-Small-Base) model and generate text locally.

This notebook demonstrates basic text continuation using the pretrained base model.

**Note:** NeoLM-Small-Base is a base language model, not an instruction-tuned assistant. Generation quality is evaluated separately in the benchmarks.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, set_seed

model_id = "../artifacts/model"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
).to(device).eval()

# Optional: enable KV Cache if loading a training checkpoint
# model.config.use_cache = True
# model.generation_config.use_cache = True

model_size = sum(p.numel() for p in model.parameters())

print("=" * 60)
print("NEOLM — INFERENCE")
print("=" * 60)
print(f"Parameters: {model_size / 1e6:.2f}M")
print(f"Device:     {device}")
print(f"Dtype:      {next(model.parameters()).dtype}")
print(f"KV Cache:   {model.generation_config.use_cache}")
print("=" * 60)

streamer = TextStreamer(
    tokenizer,
    skip_prompt=False,
    skip_special_tokens=False,
)


### Text Generation

Generate a text continuation using temperature, Top-K, Top-P, and repetition penalty.

Display the generated text and EOS stopping diagnostics.


In [ ]:
prompt = """\
Question: What is CPU stand for?"""

# set_seed(1337)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

context_limit = model.config.max_position_embeddings

prompt_length = inputs["input_ids"].shape[1]
remaining = context_limit - prompt_length

assert remaining > 0, "Prompt exceeds model context length."

max_new_tokens = min(500, remaining)

temperature = 0.3 # 0.8
top_k = 50
top_p = 0.9
repetition_penalty = 1.15

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=repetition_penalty,
    )

generated_ids = outputs[0, prompt_length:]
generated_text = tokenizer.decode(
    generated_ids,
    skip_special_tokens=False,
)

eos_id = tokenizer.eos_token_id
ended_with_eos = (
    eos_id is not None
    and len(generated_ids) > 0
    and generated_ids[-1].item() == eos_id
)

print("\n" + "=" * 55)
print(f"Prompt Tokens:       {prompt_length}")
print(f"Generated Tokens:    {len(generated_ids)}")
print(f"Total Tokens:        {outputs.shape[1]}")
print(f"EOS Token ID:        {eos_id}")
print(f"First Generated ID:  {generated_ids[0].item() if len(generated_ids) else None}")
print(f"Last Generated ID:   {generated_ids[-1].item() if len(generated_ids) else None}")
print(f"Stopped by EOS:      {ended_with_eos}")
print(f"Reached Token Limit: {len(generated_ids) == max_new_tokens and not ended_with_eos}")
print("=" * 55)